# Python — Decorators & Context Managers
**Day 3 — Python Module**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>Core Insight:</strong> Decorators are functions that wrap other functions — adding
behavior (timing, logging, retry) without modifying the original code.
Context managers guarantee cleanup (connections closed, sessions stopped) even if an exception occurs.
Both are fundamental to production Python pipelines.
</div>

In [ ]:
# Basic decorator structure — understand the pattern first
import functools

def my_decorator(func):
    @functools.wraps(func)   # preserves func.__name__, __doc__ — ALWAYS use this
    def wrapper(*args, **kwargs):
        print(f"Before {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After {func.__name__}")
        return result
    return wrapper

@my_decorator
def process_data(n: int) -> int:
    """Doubles a number."""
    return n * 2

result = process_data(5)
print(f"Result: {result}")
print(f"Function name preserved: {process_data.__name__}")   # "process_data" not "wrapper"
print(f"Docstring preserved: {process_data.__doc__}")

In [ ]:
# Timing decorator — essential for pipeline profiling
import functools
import time

def timer(func):
    """Log execution time for any function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIMER] {func.__name__}: {elapsed:.4f}s")
        return result
    return wrapper

@timer
def simulate_db_query(n_rows: int) -> list:
    """Simulate a database query with n_rows results."""
    time.sleep(0.05)   # 50ms simulated latency
    return list(range(n_rows))

@timer
def process_records(records: list) -> int:
    """Process each record."""
    return sum(r * 2 for r in records)

# Run both — timer wraps each transparently
data = simulate_db_query(1000)
total = process_records(data)
print(f"Processed {len(data)} records, total={total:,}")

In [ ]:
# Retry decorator — resilient data pipeline calls with exponential backoff
import functools, time, random

def retry(max_attempts: int = 3, delay: float = 1.0, exceptions=(Exception,)):
    """
    Retry decorator with exponential backoff.
    max_attempts: total tries (including first)
    delay: initial delay in seconds; doubles each attempt
    exceptions: tuple of exception types to catch
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts - 1:
                        raise   # final attempt — re-raise
                    wait = delay * (2 ** attempt)
                    print(f"[RETRY] {func.__name__} attempt {attempt+1} failed: {e}")
                    print(f"[RETRY] Waiting {wait:.1f}s before retry...")
                    time.sleep(wait)
        return wrapper
    return decorator

# Simulate a flaky API call
call_count = 0

@retry(max_attempts=3, delay=0.1)
def fetch_server_metrics(server_id: str) -> dict:
    global call_count
    call_count += 1
    if call_count < 3:   # fails first 2 times
        raise ConnectionError(f"Transient network error (attempt {call_count})")
    return {"server_id": server_id, "cpu": 72.5, "mem": 68.1}

result = fetch_server_metrics("srv-01")
print(f"\nSuccess: {result}")

In [ ]:
# Stacking decorators — order matters
# Applied bottom-up, executed outside-in

import functools, time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[TIMER] {func.__name__}: {time.perf_counter()-start:.4f}s")
        return result
    return wrapper

def log_args(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__}({args}, {kwargs})")
        return func(*args, **kwargs)
    return wrapper

@timer           # applied second (outer wrapper)
@log_args        # applied first (inner wrapper)
def expensive_etl(n: int, batch_size: int = 100) -> int:
    """Simulate ETL processing."""
    time.sleep(0.02)
    return n // batch_size

# Execution order: timer wraps log_args wraps expensive_etl
# Call sequence: timer.wrapper → log_args.wrapper → expensive_etl
result = expensive_etl(10000, batch_size=50)
print(f"Result: {result}")

In [ ]:
# Context Managers — guaranteed cleanup
import contextlib, time

# Method 1: class-based context manager
class DatabaseConnection:
    """Simulates a database connection with guaranteed cleanup."""
    def __init__(self, conn_string: str):
        self.conn_string = conn_string
        self.conn = None

    def __enter__(self):
        print(f"[DB] Connecting to {self.conn_string}")
        self.conn = {"connected": True, "queries": 0}   # simulated connection
        return self.conn

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[DB] Closing connection (ran {self.conn['queries']} queries)")
        self.conn = None
        if exc_type:
            print(f"[DB] Exception occurred: {exc_type.__name__}: {exc_val}")
        return False   # don't suppress exceptions

# Normal usage
with DatabaseConnection("sqlite:///telemetry.db") as db:
    db["queries"] += 1
    print(f"  Running query 1... conn={db}")
    db["queries"] += 1
    print(f"  Running query 2...")
print("Connection closed automatically.\n")

# Method 2: contextlib.contextmanager (simpler)
@contextlib.contextmanager
def timed_section(label: str):
    """Context manager for timing a block of code."""
    start = time.perf_counter()
    print(f"[TIMER] Starting: {label}")
    try:
        yield   # execution of the with block happens here
    finally:
        elapsed = time.perf_counter() - start
        print(f"[TIMER] Done: {label} ({elapsed:.4f}s)")  # always runs

with timed_section("data loading"):
    time.sleep(0.03)
    data = list(range(1000))
    print(f"  Loaded {len(data)} records")

## Interview Q&A

**Q: What does `@functools.wraps` do and why is it non-negotiable?**
A: Preserves the wrapped function's `__name__`, `__doc__`, `__module__` attributes. Without it, every decorated function shows as "wrapper" in logs, stack traces, and introspection. This breaks debugging, monitoring, and auto-generated documentation.

**Q: What are the three arguments to `__exit__`?**
A: `exc_type`, `exc_val`, `exc_tb` — exception type, value, and traceback. All are None if the with block completed normally. Return True to suppress the exception; False (or None) to re-raise it.

**Q: When would you use a retry decorator in data engineering?**
A: External API calls (Datadog, AWS S3, monitoring endpoints), database connections that may fail transiently. Use exponential backoff to avoid thundering herd. Add jitter (random noise to the delay) in production to prevent synchronized retries.

**Q: Decorator stacking order — how does it work?**
A: Decorators apply bottom-up but execute outside-in. `@timer @log_args def f()` creates `timer(log_args(f))`. When called: timer's wrapper runs first, then calls log_args's wrapper, then the original f.

**Q: Why use a context manager for a Spark session?**
A: Guarantees `spark.stop()` runs even if the job crashes mid-execution — releases cluster resources immediately. Without it, the cluster resources stay allocated until timeout (minutes to hours).

## The Citi Angle

**Production decorators at Citi:**

```python
import functools, time, logging

# 1. Combined timing + logging decorator (used on every ETL function)
def pipeline_step(step_name: str):
    """Decorator factory: logs timing and errors for each pipeline step."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            logging.info(f"STEP START: {step_name}")
            start = time.perf_counter()
            try:
                result = func(*args, **kwargs)
                elapsed = time.perf_counter() - start
                logging.info(f"STEP DONE: {step_name} ({elapsed:.2f}s)")
                return result
            except Exception as e:
                elapsed = time.perf_counter() - start
                logging.error(f"STEP FAIL: {step_name} ({elapsed:.2f}s) — {e}")
                raise
        return wrapper
    return decorator

@pipeline_step("deduplicate_telemetry")
def deduplicate(df):
    return df.drop_duplicates(subset=["server_id", "collection_date", "metric"])

@pipeline_step("compute_daily_averages")
def compute_averages(df):
    return df.groupby(["server_id", "collection_date"]).mean().reset_index()
```

**Interview line:** *"At Citi, every ETL function was wrapped with a pipeline_step decorator that provided consistent logging, timing, and error reporting. It separated cross-cutting concerns (observability) from business logic — 5 lines of decorator replaced 30+ lines of boilerplate across 50+ functions."*